In [ ]:
!pip install transformers datasets
!pip install evaluate
!pip install fsspec==2023.6.0
!pip install rouge_score
import torch
from transformers import (
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from datasets import load_dataset
import evaluate
from tqdm import tqdm
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def prepare_cnndm_dataset():
    dataset = load_dataset("cnn_dailymail", "3.0.0")

    def preprocess_function(examples):
        inputs = examples["article"]
        targets = examples["highlights"]
        return {"inputs": inputs, "targets": targets}

    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["article", "highlights", "id"]
    )
    return dataset["train"], dataset["validation"], dataset["test"]

train_dataset, val_dataset, test_dataset = prepare_cnndm_dataset()

model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)

def data_collator(batch):
    inputs = [item["inputs"] for item in batch]
    targets = [item["targets"] for item in batch]

    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        ).input_ids

    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels
    }

def train_model(model, train_dataset, val_dataset, epochs=1, batch_size=4):
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=100,
        num_training_steps=len(train_loader) * epochs
    )

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc="Training"):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Average Loss: {avg_loss:.4f}")

    return model


print("\nFine-tuning PEGASUS on CNN/DailyMail...")
model = train_model(
    model,
    train_dataset.select(range(1000)),  # 1000 samples
    val_dataset.select(range(200)),     # 200 samples
    epochs=3,
    batch_size=4
)


rouge = evaluate.load("rouge")

def evaluate_model(model, test_dataset, num_samples=100):
    test_subset = test_dataset.select(range(num_samples))
    predictions, references = [], []

    model.eval()
    with torch.no_grad():
        for example in tqdm(test_subset, desc="Evaluating"):
            inputs = tokenizer(
                example["inputs"],
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            ).to(device)

            summary_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=256,
                num_beams=4,
                early_stopping=True
            )

            predictions.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True))
            references.append(example["targets"])

    return rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )


print("\nEvaluating PEGASUS...")
results = evaluate_model(model, test_dataset, num_samples=50)


results_df = pd.DataFrame({
    "Model": ["PEGASUS-cnn_dailymail"],
    "ROUGE-1": [results["rouge1"]],
    "ROUGE-2": [results["rouge2"]],
    "ROUGE-L": [results["rougeL"]]
})

print("\nEvaluation Results:")
print(results_df)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have 

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

...

Training:   0%|          | 0/250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Training:   0%|          | 1/250 [00:02<12:18,  2.97s/it]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Training:   1%|          | 2/250 [00:05<12:13,  2.96s/it]/usr/local/lib/python3.11/dist-packag

Average Loss: 0.7653



Evaluating PEGASUS...


Evaluating: 100%|██████████| 50/50 [01:12<00:00,  1.45s/it]



Evaluation Results:
                   Model  ROUGE-1   ROUGE-2   ROUGE-L
0  PEGASUS-cnn_dailymail  0.36548  0.153281  0.262121
